# 056 — Simple RNN: Architecture and Forward Propagation

The whole architecture is one line:

$$h_t = \tanh(W_x x_t + W_h h_{t-1} + b)$$

Implemented from scratch below with $F=5$, $H=3$: **27** parameters in the
recurrent layer, **31** in the whole model — and a perturbation test showing
the final state really does carry the first word.

| Part | What we check |
|---|---|
| A | the forward pass, step by step |
| B | the parameter count, and no sequence length in it |
| C | the same weights at every step |
| D | the memory, tested by perturbation |

## Part A — Three steps of one cell

In [ ]:
import numpy as np

T, F, H = 3, 5, 3                    # steps, features, hidden units
rng = np.random.default_rng(1)
Wx = rng.normal(0, 0.5, (F, H))      # input weights
Wh = rng.normal(0, 0.5, (H, H))      # recurrent weights - an ANN has no analogue
b = np.zeros(H)
Wo, bo = rng.normal(0, 0.5, (H, 1)), 0.0

X = np.eye(F)[[0, 1, 2]]             # three one-hot "words"

h = np.zeros(H)                      # the initial state is zeros
for t in range(T):
    h = np.tanh(X[t] @ Wx + h @ Wh + b)
    print(f"t={t + 1}  h = [{', '.join(f'{v:+.4f}' for v in h)}]")

pred = 1 / (1 + np.exp(-(h @ Wo + bo)))
print(f"prediction {pred[0]:.4f}")
assert h.shape == (H,)

## Part B — $(F + H)H + H$, with no $T$ in it

In [ ]:
formula = (F + H) * H + H
print(f"input weights   {F}x{H} = {F * H}")
print(f"recurrent       {H}x{H} = {H * H}      <- the whole difference from an ANN")
print(f"biases                  {H}")
print(f"RNN layer = {formula}, output layer = {H + 1}, total = {formula + H + 1}")
assert formula == Wx.size + Wh.size + b.size == 27
assert formula + H + 1 == 31
print()
print("No T anywhere. Ten steps or ten thousand, the same 27 parameters.")
for steps in (3, 30, 3000):
    assert (F + H) * H + H == 27
print("checked at 3, 30 and 3000 steps")

## Part C — The same weights, every step

Unrolling a diagram three times does not create three layers.

In [ ]:
seen = []
h = np.zeros(H)
for t in range(T):
    seen.append((Wx.copy(), Wh.copy()))
    h = np.tanh(X[t] @ Wx + h @ Wh + b)

assert all(np.array_equal(seen[0][0], w) for w, _ in seen)
assert all(np.array_equal(seen[0][1], w) for _, w in seen)
print("the weight matrices at step 1, 2 and 3 are byte-identical")
print()
print("What changes between steps is the state carried in, not the weights.")
print(f"and the state is the only thing with memory in it: shape {h.shape}")

## Part D — Does the last state really remember the first word?

Change only word 1 and see whether the final state moves.

In [ ]:
def forward(sequence):
    state = np.zeros(H)
    for step in sequence:
        state = np.tanh(step @ Wx + state @ Wh + b)
    return state

altered = X.copy()
altered[0] = np.eye(F)[3]            # a different FIRST word; words 2 and 3 unchanged
shift = np.abs(forward(X) - forward(altered)).max()
print(f"changing only word 1 moves the final state by {shift:.4f}")
assert round(shift, 4) == 0.0192
assert shift > 0
print("the last hidden state carries the first input. That is the memory.")

# and it fades: the same change at the LAST step moves it much further
altered_last = X.copy()
altered_last[-1] = np.eye(F)[3]
late = np.abs(forward(X) - forward(altered_last)).max()
print(f"the same change at the last step moves it {late:.4f} "
      f"({late / shift:.0f}x more)")
assert late > shift
print("recent words dominate - which is the vanishing-memory problem that")
print("LSTMs exist to fix.")

## What this notebook established

- $h_t = \tanh(W_x x_t + W_h h_{t-1} + b)$ run by hand for three steps.
- $(F + H)H + H$ = **27** parameters for $F=5$, $H=3$; **31** with the output
  layer. No sequence length appears in the formula.
- The weight matrices are identical at every step — unrolling is a drawing
  device, not three layers.
- Changing the first word moved the final state by **0.0192**; the same change
  at the last step moved it much further, so the memory of early inputs fades.

## Exercises

1. Run 20 steps and plot the perturbation from step 1 against step 20. Where
   does the influence become invisible?
2. Scale `Wh` by 2.0 and re-run. What happens to `tanh`, and to the memory?
3. Add a second recurrent layer that consumes every hidden state. How many
   parameters now?